In [42]:
%pip install python-dotenv langchainhub

import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
# Already imported in a later notebook cell; no duplicate import needed.
import langchain

import requests


Note: you may need to restart the kernel to use updated packages.


In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [44]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults


os.environ["TAVILY_API_KEY"] = "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"

search_tool = TavilySearchResults(
    max_results=2, 
    tavily_api_key=os.environ["TAVILY_API_KEY"]
)


In [45]:

def get_weather_data(city: str) -> str:
    """Get the current weather for a given city."""
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"could not fetch weather data for {city}"

    return (
        f"The current temperature in {city} is {data['current']['temperature']}°C, "
        f"with {data['current']['weather_descriptions'][0].lower()} and "
        f"humidity of {data['current']['humidity']}%."
    )

In [46]:

resultados = search_tool.invoke("What is the capital of France?")
resultados

[{'title': 'What is the capital city of France?',
  'url': 'https://www.facebook.com/groups/376100836095453/posts/2780652808973565',
  'content': 'Paris[a] is the capital and largest city of France. With an official estimated population of 2,102,650 residents as of 1 January 2023[2] in an',
  'score': 0.93554133},
 {'title': 'What is the capital city of France and why is it famous? - Brainly.in',
  'url': 'https://brainly.in/question/62118432',
  'content': 'Brainly.in\n\n# What is the capital city of France and why is it famous?\n\nAnswer:\n\nThe capital city of France is Paris.\n\nIt’s famous for several reasons:\n\nCultural heritage – Paris is home to iconic landmarks like the Eiffel Tower, Louvre Museum (housing the Mona Lisa), and Notre-Dame Cathedral.\n\nFashion capital – Known as the global hub for fashion and luxury brands, hosting events like Paris Fashion Week.\n\nHistory – The city has played a major role in European history, from the French Revolution to modern politics.\n\

In [ ]:
from anthropic import Anthropic

client = Anthropic(api_key= os.getenv("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "What year is it?"}]
)

for block in response.content:
    if block.type == "text":
        print(block.text)
        break


I don't have access to real-time information, so I can't tell you the current date with certainty. My knowledge has a training cutoff, and I don't have a live clock or calendar connection.

If you let me know the current date, I'm happy to work with that information! Or if you're asking for another reason, let me know how I can help.


In [48]:
# from langchain import hub
# prompt = hub.pull("hwchase17/react")

In [49]:
from langchain_core.prompts import PromptTemplate

template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [50]:
tools = [search_tool, get_weather_data]

In [51]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
%pip install -q langchain-anthropic

from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    api_key=os.getenv("ANTHROPIC_API_KEY")
)



Note: you may need to restart the kernel to use updated packages.


In [53]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="What years it is?"
)

In [54]:
result = agent.invoke({"messages": [("user", "find the capital of india" "and then find its current weather.")]})
final_message = result["messages"][-1]
print(final_message.content)

Here's a summary of the findings:

🇮🇳 **Capital of India:** New Delhi
- Located in the north-central part of India, west of the Yamuna River.
- It became the capital in 1911, replacing Calcutta (now Kolkata).

🌤️ **Current Weather in New Delhi:**
- 🌡️ **Temperature:** 32°C
- ☀️ **Condition:** Clear
- 💧 **Humidity:** 35%

It's a warm and clear day in New Delhi! Perfect weather to explore the city! 😊
